# 🏋️ LoRA Training Demo

This notebook demonstrates LoRA (Low-Rank Adaptation) training using a pre-prepared dataset of Socratic tutoring examples.

> ⚠️ **Workshop Scope:** We'll train a tiny model (`Qwen2-0.5B`) on CPU with the pre-prepared training data. This demonstrates the *process*—production training uses larger models on GPU with more data.

**Training Data:** We use `socratic-training-data-curated.jsonl` - a curated dataset of 30 high-quality Socratic tutoring examples where the tutor responds with guiding questions rather than direct answers.

You'll:
1. Load the pre-prepared training data
2. Configure LoRA parameters
3. **Actually train** a LoRA adapter (takes ~5-10 minutes on CPU)
4. Save the trained adapter

**Next:** Run `3-evaluation.ipynb` to test and compare the trained model.

## What is LoRA?

Full fine-tuning updates **every** parameter in the model. For a 3B model, that's 3 billion weights.

LoRA freezes the original model and trains small "adapter" matrices that modify behavior. Think of it like adding a filter to a camera—the camera stays the same, but the output changes.

| Approach | Parameters Trained | Storage | Memory |
|----------|-------------------|---------|--------|
| Full fine-tuning | 3B (100%) | ~6 GB | 24+ GB GPU |
| LoRA | 15M (~0.5%) | ~30 MB | 8-16 GB GPU |
| QLoRA (4-bit) | 15M (~0.5%) | ~30 MB | 4-8 GB GPU |

---
## 0. Setup

We'll use the same small model from the model-optimization module.

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Same small model used in model-optimization module
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
OUTPUT_DIR = "./socratic-tutor-lora"
TRAINING_DATA = "socratic-training-data-curated.jsonl"  # Pre-prepared curated dataset

print(f"Base model: {MODEL_ID}")
print(f"Training data: {TRAINING_DATA}")
print(f"Output directory: {OUTPUT_DIR}")

---
## 1. Load Training Data

We use a pre-prepared dataset of 30 high-quality Socratic tutoring examples. Each example shows:
- A student asking a math question
- A tutor responding with **guiding questions** (never direct answers)

This is the kind of curated dataset you would create in production—either by hand or by filtering LLM-generated data for quality.

In [ ]:
import json

# Load the pre-prepared training data
training_examples = []
with open(TRAINING_DATA, "r") as f:
    for line in f:
        if line.strip():  # Skip empty lines
            training_examples.append(json.loads(line))

print(f"✅ Loaded {len(training_examples)} training examples from {TRAINING_DATA}")
print(f"\nExample conversations:")
for i, ex in enumerate(training_examples[:3]):
    print(f"\n--- Example {i+1} ---")
    print(f"  Student: {ex['messages'][0]['content']}")
    print(f"  Tutor: {ex['messages'][1]['content']}")

In [ ]:
# Format data for training (convert messages format to text)
from datasets import Dataset

# System prompt that defines Socratic behavior - this gets baked into every example
SYSTEM_PROMPT = """You are a Socratic math tutor. Never give direct answers. Instead:
- Ask guiding questions to help students discover the solution
- Encourage critical thinking and problem-solving
- Be supportive and patient"""

def format_for_training(example):
    """Format example as a chat conversation for Qwen with system prompt."""
    user_msg = example['messages'][0]['content']
    assistant_msg = example['messages'][1]['content']
    # Include system prompt in training to reinforce behavior
    return {
        "text": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    }

# Create dataset with data augmentation (repeat examples to increase training signal)
formatted_data = []
AUGMENTATION_FACTOR = 3  # Repeat each example 3 times

for example in training_examples:
    for _ in range(AUGMENTATION_FACTOR):
        formatted_data.append(format_for_training(example))

train_dataset = Dataset.from_list(formatted_data)

print(f"Original examples: {len(training_examples)}")
print(f"After {AUGMENTATION_FACTOR}x augmentation: {len(train_dataset)} training samples")
print(f"\nFormatted example:")
print(train_dataset[0]['text'])

---
## 2. Load Base Model

Load the small Qwen model. This is the same model used in the model-optimization module.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading base model (this may take a minute)...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Load model on CPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,  # Use float32 for CPU
    device_map="cpu"
)

print(f"✅ Model loaded: {MODEL_ID}")
print(f"   Parameters: {model.num_parameters():,}")

---
## 3. Configure LoRA

Set up the LoRA adapter configuration.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# LoRA configuration - small rank for quick training
lora_config = LoraConfig(
    r=8,                      # Rank - small for CPU demo
    lora_alpha=16,            # Scaling factor (usually 2×r)
    lora_dropout=0.1,         # Regularization
    target_modules=[          # Which layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",
    ],
    task_type=TaskType.CAUSAL_LM,
)

print("LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Target modules: {lora_config.target_modules}")

In [ ]:
# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Show trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

---
## 4. Train the Model

Now let's actually train using our curated data! This will take **5-10 minutes** on CPU.

> ☕ Good time for a coffee break!

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

print(f"Tokenized dataset size: {len(tokenized_dataset)}")

In [ ]:
# Training arguments - optimized for effective learning on small dataset
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,             # More epochs for small dataset
    per_device_train_batch_size=2,   # Slightly larger batch
    gradient_accumulation_steps=4,   # Accumulate gradients
    learning_rate=2e-4,              # Slightly higher LR for faster learning
    warmup_ratio=0.1,                # 10% warmup
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,                      # No FP16 on CPU
    optim="adamw_torch",
    report_to="none",                # Disable wandb etc.
    lr_scheduler_type="cosine",      # Cosine schedule works well for fine-tuning
)

print("Training Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  LR scheduler: {training_args.lr_scheduler_type}")

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print(f"Training on {len(train_dataset)} samples...")
print("This will take approximately 5-10 minutes on CPU.\n")

# Train!
trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")

---
## 5. Save the Adapter

Save the trained LoRA adapter for evaluation and deployment.

In [ ]:
# Save the adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Adapter saved to: {OUTPUT_DIR}")

# Show what was saved
print("\nSaved files:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    if size > 1024*1024:
        print(f"  {f}: {size/1024/1024:.1f} MB")
    elif size > 1024:
        print(f"  {f}: {size/1024:.1f} KB")
    else:
        print(f"  {f}: {size} B")

---
## Summary

You've completed an actual LoRA fine-tuning run! Here's what you did:

1. ✅ **Loaded training data** — 30 curated Socratic tutoring examples
2. ✅ **Configured LoRA** — Set rank, alpha, and target modules
3. ✅ **Trained an adapter** — Ran actual training on CPU
4. ✅ **Saved the adapter** — Ready for evaluation

### The Full Pipeline

```
Notebook 1: Generate Data    →    Notebook 2: Train Model    →    Notebook 3: Evaluate
(socratic-training-data.jsonl)    (socratic-tutor-lora/)          (compare before/after)
```

### Key Takeaways

- **LoRA is efficient** — Only ~0.5% of parameters trained
- **Adapters are small** — Just a few MB vs. GB for full model
- **Same process at any scale** — CPU demo → GPU production

### Next Step

Run **`3-evaluation.ipynb`** to compare the base model vs fine-tuned model and see if training made a difference!